In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
YIANNIS PROTOCOL — FAST ROCKET-ONLY + QUIET SCREEN LOGS (FULL UPDATED)

✅ Rocket-only (NO Crane)
✅ Hard time limit per experiment (default 120s)
✅ Sweep lambda over multiple values for each objective:
     - baseline (paper Rocket objective)
     - hinge2 stability penalty (Option A)
     - softplus stability penalty (Option B)
✅ Save ALL solutions:
     - per-run ranking CSV: ["Node ID","Order"]
     - ALL_RUNS.csv + AGG.csv
✅ NO per-run log files; prints progress to screen (QUIET / THROTTLED)

WHY STABILITY IS SLOWER:
- baseline Rocket per-iter is O(m) with simple ops + one scatter
- stability adds multiple np.bincount reductions over m edges (big constant factor)
This script includes instrumentation and throttles:
  - stab_every: compute stability gradient only every K iters
  - eval_every: compute discrete FW only every K iters (argsort + FW)
  - print_every_sec + min_print_iter_gap + print_on_improve_only: reduce console spam

NOTE (Yiannis / Ioannis):
- Crane is NOT used (Rocket-only), consistent with "crane is not needed".
- The ONLY algorithmic difference across objectives is the objective/gradient:
    baseline: maximize sum(w_hat * sigmoid(beta*(P[v]-P[u])))
    hinge2/softplus: baseline + lambda * stability_penalty(P)
  All other mechanics (init, Adam, checkpoints, output CSV format) are identical.

Dependencies: numpy, pandas
"""

import os
import time
from collections import defaultdict

import numpy as np
import pandas as pd


# ============================================================
# 0) Helpers
# ============================================================

def safe_mkdir(path: str):
    os.makedirs(path, exist_ok=True)
    return path

def fmt_lam(lam: float):
    s = f"{lam:.6g}"
    return s.replace(".", "p").replace("-", "m")

def ratio_percent(fw, total_w):
    return (100.0 * fw / total_w) if total_w > 0 else 0.0

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def softplus(x):
    # stable softplus
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0.0)

def write_ranking_csv_nodeid_order(path, index_to_node, rank_arr):
    n = len(rank_arr)
    rows = [{"Node ID": str(index_to_node[i]).strip(), "Order": int(rank_arr[i])} for i in range(n)]
    rows.sort(key=lambda r: r["Order"])
    pd.DataFrame(rows).to_csv(path, index=False)


# ============================================================
# 1) DIMACS reader (aggregates parallel arcs) -> deterministic
# ============================================================

def read_graph_dimacs_agg(file_path: str):
    agg = defaultdict(float)
    node_ids = set()

    with open(file_path, "r") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith(("c", "p")):
                continue
            if not line.startswith("a"):
                continue

            parts = line.split()
            if len(parts) < 4:
                continue

            u = parts[1]
            v = parts[2]
            try:
                w = float(parts[3])
            except ValueError:
                continue

            node_ids.add(u)
            node_ids.add(v)
            agg[(u, v)] += w

    node_list = sorted(node_ids)
    node_to_index = {node: i for i, node in enumerate(node_list)}
    index_to_node = {i: node for node, i in node_to_index.items()}

    edges_indexed = [(node_to_index[u], node_to_index[v], float(w_sum))
                     for (u, v), w_sum in agg.items()]
    edges_indexed.sort(key=lambda e: (e[0], e[1]))
    return edges_indexed, node_to_index, index_to_node


# ============================================================
# 2) Discrete objective: forward/backward from arrays + rank
# ============================================================

def compute_forward_backward_from_arrays(eu, ev, ew, rank_arr):
    total_w = float(np.sum(ew))
    fw = float(np.sum(ew[rank_arr[eu] < rank_arr[ev]]))
    bw = total_w - fw
    return total_w, fw, bw


# ============================================================
# 3) Stability penalty (Option A/B) + d(L_stab)/d d_e
#     d_e := P[v] - P[u]
# ============================================================

def stability_loss_and_edge_dLd_d_bincount(
    n: int,
    eu: np.ndarray, ev: np.ndarray, ew: np.ndarray,
    P: np.ndarray,
    beta: float,
    loss_type: str,          # "hinge2" | "softplus"
    normalize: bool = True,
    eps_norm: float = 1e-12,
    margin: float = 0.0,
):
    d = (P[ev] - P[eu]).astype(np.float32)
    x = (float(beta) * d).astype(np.float32)

    fwd = sigmoid(x).astype(np.float32)
    bwd = (1.0 - fwd).astype(np.float32)

    w = ew.astype(np.float64)

    # 4 bincount reductions over edges
    BackIn  = np.bincount(ev, weights=w * bwd.astype(np.float64), minlength=n)
    FwdOut  = np.bincount(eu, weights=w * fwd.astype(np.float64), minlength=n)
    BackOut = np.bincount(eu, weights=w * bwd.astype(np.float64), minlength=n)
    FwdIn   = np.bincount(ev, weights=w * fwd.astype(np.float64), minlength=n)

    v1 = BackIn - FwdOut
    v2 = BackOut - FwdIn

    if normalize:
        Tot = np.bincount(eu, weights=w, minlength=n) + np.bincount(ev, weights=w, minlength=n)
        denom = Tot + float(eps_norm)
        v1n = v1 / denom
        v2n = v2 / denom
    else:
        denom = None
        v1n, v2n = v1, v2

    if loss_type == "hinge2":
        r1 = np.maximum(v1n, 0.0)
        r2 = np.maximum(v2n, 0.0)
        L = float(np.sum(r1 * r1) + np.sum(r2 * r2))

        dv1n = (2.0 * r1).astype(np.float64)
        dv2n = (2.0 * r2).astype(np.float64)

        if normalize:
            dv1 = dv1n / denom
            dv2 = dv2n / denom
        else:
            dv1, dv2 = dv1n, dv2n

    elif loss_type == "softplus":
        z1 = v1n - float(margin)
        z2 = v2n - float(margin)
        L = float(np.sum(softplus(z1)) + np.sum(softplus(z2)))

        s1 = sigmoid(z1).astype(np.float64)
        s2 = sigmoid(z2).astype(np.float64)

        if normalize:
            dv1 = s1 / denom
            dv2 = s2 / denom
        else:
            dv1, dv2 = s1, s2
    else:
        raise ValueError(f"Unknown stability loss_type={loss_type}")

    # Vertex coeffs for aggregates
    g_BackIn  = dv1
    g_FwdOut  = -dv1
    g_BackOut = dv2
    g_FwdIn   = -dv2

    # df/dd = beta*fwd*(1-fwd); db/dd = -df/dd
    df_dd = (float(beta) * (fwd * (1.0 - fwd))).astype(np.float32)
    db_dd = (-df_dd).astype(np.float32)

    edge_dLd_d = (
        (ew * db_dd) * (g_BackIn[ev].astype(np.float32) + g_BackOut[eu].astype(np.float32)) +
        (ew * df_dd) * (g_FwdOut[eu].astype(np.float32) + g_FwdIn[ev].astype(np.float32))
    ).astype(np.float32)

    return L, edge_dLd_d


# ============================================================
# 4) Rocket with instrumentation + throttles + QUIET printing
# ============================================================

def rocket_optimize_objective(
    n: int,
    eu: np.ndarray, ev: np.ndarray, ew: np.ndarray,
    *,
    beta: float,
    lr: float,
    seed: int,
    deadline: float,
    objective: str,          # "baseline" | "hinge2" | "softplus"
    lambda_stab: float,
    stab_normalize: bool,
    stab_margin: float,
    total_w: float,
    eval_every: int = 1000,          # argsort+FW checkpoint interval
    stab_every: int = 5,             # compute stability gradient every K iters (1 = every iter)
    print_every_sec: float = 10.0,   # rate-limit prints
    min_print_iter_gap: int = 300,   # don't print again unless >= this many iters passed
    print_on_improve_only: bool = True,  # mostly print when best improves
):
    rng = np.random.default_rng(seed)
    P = rng.standard_normal(n).astype(np.float32)

    # max-normalize weights (paper Eq. 7)
    wmax = float(np.max(ew)) if ew.size else 1.0
    if wmax <= 0:
        wmax = 1.0
    w_hat = (ew / wmax).astype(np.float32)

    # Adam state
    m_adam = np.zeros(n, dtype=np.float32)
    v_adam = np.zeros(n, dtype=np.float32)
    b1, b2 = 0.9, 0.999
    eps = 1e-8
    t = 0

    def rank_from_positions(Pvec):
        perm = np.argsort(Pvec, kind="mergesort").astype(np.int32)
        r = np.empty(n, dtype=np.int32)
        r[perm] = np.arange(n, dtype=np.int32)
        return r

    # initial discrete best
    rank0 = rank_from_positions(P)
    fw_best = float(np.sum(ew[rank0[eu] < rank0[ev]]))
    rank_best = rank0.copy()

    # stats
    it = 0
    last_print_t = time.time()
    last_print_it = 0
    last_reported_best = fw_best

    stab_calls = 0
    stab_time = 0.0
    base_time = 0.0
    eval_calls = 1
    eval_time = 0.0

    last_L_stab = 0.0

    while time.time() < deadline:
        it += 1

        tb0 = time.time()
        d = (P[ev] - P[eu]).astype(np.float32)
        x = (float(beta) * d).astype(np.float32)
        sig_f = sigmoid(x).astype(np.float32)
        sigp = (sig_f * (1.0 - sig_f)).astype(np.float32)

        edge_grad_u = (w_hat * sigp * float(beta)).astype(np.float32)
        base_time += (time.time() - tb0)

        # stability gradient (throttled)
        if objective != "baseline" and float(lambda_stab) != 0.0:
            if stab_every <= 1 or (it % int(stab_every) == 0):
                ts0 = time.time()
                L_stab, edge_dLd_d = stability_loss_and_edge_dLd_d_bincount(
                    n=n, eu=eu, ev=ev, ew=ew,
                    P=P, beta=float(beta),
                    loss_type=str(objective),
                    normalize=bool(stab_normalize),
                    margin=float(stab_margin),
                )
                edge_grad_u = (edge_grad_u - float(lambda_stab) * edge_dLd_d).astype(np.float32)
                stab_time += (time.time() - ts0)
                stab_calls += 1
                last_L_stab = float(L_stab)
        else:
            last_L_stab = 0.0

        # scatter + Adam
        grad = np.zeros(n, dtype=np.float32)
        np.add.at(grad, eu, +edge_grad_u)
        np.add.at(grad, ev, -edge_grad_u)

        t += 1
        m_adam = (b1 * m_adam + (1 - b1) * grad).astype(np.float32)
        v_adam = (b2 * v_adam + (1 - b2) * (grad * grad)).astype(np.float32)
        mhat = m_adam / (1 - (b1 ** t))
        vhat = v_adam / (1 - (b2 ** t))
        P = (P - float(lr) * mhat / (np.sqrt(vhat) + eps)).astype(np.float32)

        # discrete FW checkpoint (expensive because argsort)
        if eval_every and (it % int(eval_every) == 0):
            te0 = time.time()
            rank = rank_from_positions(P)
            fw = float(np.sum(ew[rank[eu] < rank[ev]]))
            eval_time += (time.time() - te0)
            eval_calls += 1
            if fw > fw_best + 1e-9:
                fw_best = fw
                rank_best = rank.copy()

        # QUIET PRINT: time + iter gap, and (optionally) only on improvement
        now = time.time()
        time_ok = (now - last_print_t) >= float(print_every_sec)
        iter_ok = (it - last_print_it) >= int(min_print_iter_gap)
        improved = (fw_best > last_reported_best + 1e-9)

        do_print = False
        if print_on_improve_only:
            if improved:
                do_print = True
            elif time_ok and iter_ok:
                # rare heartbeat
                do_print = True
        else:
            if time_ok and iter_ok:
                do_print = True

        if do_print:
            dt = now - last_print_t
            dits = it - last_print_it
            ips = (dits / dt) if dt > 1e-9 else 0.0
            remaining = max(0.0, deadline - now)

            best_ratio = ratio_percent(fw_best, total_w)

            stab_avg = (stab_time / max(1, stab_calls)) if stab_calls else 0.0
            eval_avg = (eval_time / max(1, eval_calls)) if eval_calls else 0.0
            note = "IMPROVED" if improved else "..."

            print(
                f"[rocket] {note} obj={objective:<8} lam={lambda_stab:g} it={it} ips={ips:.1f} "
                f"best={best_ratio:.4f}% rem={remaining:.1f}s "
                f"stab_calls={stab_calls} stab_avg={stab_avg*1000:.2f}ms "
                f"eval_avg={eval_avg*1000:.2f}ms "
                f"L_stab(last)={last_L_stab:.3g}"
            )

            last_print_t = now
            last_print_it = it
            last_reported_best = fw_best

    # final checkpoint
    rank = rank_from_positions(P)
    fw = float(np.sum(ew[rank[eu] < rank[ev]]))
    if fw > fw_best + 1e-9:
        fw_best = fw
        rank_best = rank.copy()

    return rank_best, fw_best, it, {
        "stab_calls": stab_calls,
        "stab_time_s": float(stab_time),
        "base_time_s": float(base_time),
        "eval_calls": eval_calls,
        "eval_time_s": float(eval_time),
    }


# ============================================================
# 5) One run (Rocket-only) — QUIET screen logs
# ============================================================

def run_one_config(
    *,
    dimacs_path: str,
    index_to_node,
    n: int,
    eu, ev, ew,
    out_csv_path: str,
    seed: int,
    time_limit_s: float,
    rocket_beta: float,
    rocket_lr: float,
    objective: str,          # "baseline"|"hinge2"|"softplus"
    lambda_stab: float,
    stab_normalize: bool,
    stab_margin: float,
    eval_every: int,
    stab_every: int,
    print_every_sec: float,
    min_print_iter_gap: int,
    print_on_improve_only: bool,
    print_run_start_done: bool,
):
    t0 = time.time()
    deadline = t0 + float(time_limit_s)
    total_w = float(np.sum(ew))

    if print_run_start_done:
        print(f"\n[run] START obj={objective} lam={lambda_stab:g} seed={seed} budget={time_limit_s}s "
              f"eval_every={eval_every} stab_every={stab_every}")

    tR0 = time.time()
    rank_best, fw_best, iters_done, stats = rocket_optimize_objective(
        n=int(n), eu=eu, ev=ev, ew=ew,
        beta=float(rocket_beta),
        lr=float(rocket_lr),
        seed=int(seed),
        deadline=float(deadline),
        objective=str(objective),
        lambda_stab=float(lambda_stab),
        stab_normalize=bool(stab_normalize),
        stab_margin=float(stab_margin),
        total_w=total_w,
        eval_every=int(eval_every),
        stab_every=int(stab_every),
        print_every_sec=float(print_every_sec),
        min_print_iter_gap=int(min_print_iter_gap),
        print_on_improve_only=bool(print_on_improve_only),
    )
    rocket_s = time.time() - tR0

    tot1, fw1, bw1 = compute_forward_backward_from_arrays(eu, ev, ew, rank_best)
    write_ranking_csv_nodeid_order(out_csv_path, index_to_node, rank_best)

    elapsed = time.time() - t0
    final_ratio = ratio_percent(float(fw1), float(tot1))

    if print_run_start_done:
        print(
            f"[run] DONE obj={objective} lam={lambda_stab:g} seed={seed} "
            f"final={final_ratio:.6f}% rocket_s={rocket_s:.2f} total_s={elapsed:.2f} "
            f"iters={iters_done} "
            f"stab_calls={stats['stab_calls']} stab_time_s={stats['stab_time_s']:.2f} "
            f"eval_time_s={stats['eval_time_s']:.2f} "
            f"csv={out_csv_path}"
        )

    return {
        "seed": int(seed),
        "n": int(n),
        "m": int(eu.size),
        "objective": str(objective),
        "lambda_stab": float(lambda_stab),
        "stab_normalize": bool(stab_normalize),
        "stab_margin": float(stab_margin),
        "eval_every": int(eval_every),
        "stab_every": int(stab_every),

        "ratio_final_%": float(final_ratio),
        "final_fw": float(fw1),
        "final_bw": float(bw1),
        "total_w": float(tot1),

        "iters_done": int(iters_done),
        "t_rocket_s": float(rocket_s),
        "t_total_s": float(elapsed),

        "stab_calls": int(stats["stab_calls"]),
        "stab_time_s": float(stats["stab_time_s"]),
        "eval_calls": int(stats["eval_calls"]),
        "eval_time_s": float(stats["eval_time_s"]),
        "base_time_s": float(stats["base_time_s"]),

        "out_csv": out_csv_path,
    }


# ============================================================
# 6) Main — Yiannis protocol sweep (Rocket-only)
# ============================================================

if __name__ == "__main__":
    # ------------------------------
    # INPUT
    # ------------------------------
    edge_file = "/mmfs1/home/sv96/Feedback-arc-set-paper/datasets/connectome.d"

    # ------------------------------
    # OUTPUT to Desktop (fallback if not exists)
    # ------------------------------
    desktop = os.path.join(os.path.expanduser("~"), "Desktop")
    out_root = desktop if os.path.isdir(desktop) else os.path.dirname(os.path.abspath(edge_file))
    out_dir = safe_mkdir(os.path.join(out_root, "yiannis_rocket_only_outputs"))

    base = os.path.basename(edge_file)
    base = base[:-2] if base.endswith(".d") else base

    # ------------------------------
    # Yiannis protocol budgets
    # ------------------------------
    TIME_LIMIT_S = 120.0

    # Seeds
    SEEDS = [1, 2, 3]
    # SEEDS = [1, 2, 3, 4, 5]

    # Rocket hyperparams
    ROCKET_BETA = 1.0
    ROCKET_LR = 0.05

    # Speed knobs
    EVAL_EVERY = 1000        # argsort+FW frequency (lower = more accurate tracking but slower)
    STAB_EVERY = 5           # stability gradient frequency (1 = slowest / most faithful)
    STAB_MARGIN = 0.0

    # QUIET printing knobs
    PRINT_EVERY_SEC = 10.0        # was 2.0
    MIN_PRINT_ITER_GAP = 300      # don't print too often even if time passes
    PRINT_ON_IMPROVE_ONLY = True  # print mainly when best improves
    PRINT_RUN_HEADERS = True      # "===== RUN i/j ..." lines
    PRINT_RUN_START_DONE = True   # prints START/DONE per run (set False for even quieter)

    # Objectives (Ioannis options)
    OBJECTIVES = [
        ("baseline", False),
        ("hinge2", True),
        ("softplus", True),
    ]

    # Lambda sweep (baseline forces lambda=0)
    LAM_STAB_LIST = [0.0, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]

    # ------------------------------
    # Load graph once
    # ------------------------------
    t_load0 = time.time()
    edges_indexed, node_to_index, index_to_node = read_graph_dimacs_agg(edge_file)
    n = len(node_to_index)
    m = len(edges_indexed)

    eu = np.empty(m, dtype=np.int32)
    ev = np.empty(m, dtype=np.int32)
    ew = np.empty(m, dtype=np.float32)
    for i, (u, v, w) in enumerate(edges_indexed):
        eu[i] = u
        ev[i] = v
        ew[i] = float(w)

    total_w = float(np.sum(ew))
    load_s = time.time() - t_load0

    print("\n================= YIANNIS PROTOCOL (ROCKET-ONLY) =================")
    print(f"Input:      {edge_file}")
    print(f"Out dir:    {out_dir}")
    print(f"Loaded:     n={n} m={m} total_w={total_w:.6f} load_s={load_s:.2f}")
    print(f"Budget:     time_limit_s={TIME_LIMIT_S}  beta={ROCKET_BETA}  lr={ROCKET_LR}")
    print(f"Speed:      eval_every={EVAL_EVERY}  stab_every={STAB_EVERY}")
    print(f"Logs:       print_every_sec={PRINT_EVERY_SEC}  min_print_iter_gap={MIN_PRINT_ITER_GAP}  "
          f"improve_only={int(PRINT_ON_IMPROVE_ONLY)}")
    print(f"Seeds:      {SEEDS}")
    print(f"Lambdas:    {LAM_STAB_LIST} (baseline forces lambda=0)")
    print("------------------------------------------------------------------")

    # Progress count
    total_runs = 0
    for obj, _norm in OBJECTIVES:
        total_runs += len(SEEDS) * (1 if obj == "baseline" else len(LAM_STAB_LIST))

    rid = 0
    t_sweep0 = time.time()
    all_rows = []

    for objective, stab_norm in OBJECTIVES:
        lam_list = [0.0] if objective == "baseline" else LAM_STAB_LIST
        for lam in lam_list:
            for seed in SEEDS:
                rid += 1
                tag = f"{objective}_norm{int(stab_norm)}_lamSTAB{fmt_lam(lam)}"
                out_csv = os.path.join(out_dir, f"{base}_Seke_{tag}_seed{seed}_ranking.csv")

                if PRINT_RUN_HEADERS:
                    print(f"\n===== RUN {rid}/{total_runs} : obj={objective}, lam={lam}, seed={seed} =====")

                res = run_one_config(
                    dimacs_path=edge_file,
                    index_to_node=index_to_node,
                    n=n,
                    eu=eu, ev=ev, ew=ew,
                    out_csv_path=out_csv,
                    seed=int(seed),
                    time_limit_s=float(TIME_LIMIT_S),
                    rocket_beta=float(ROCKET_BETA),
                    rocket_lr=float(ROCKET_LR),
                    objective=str(objective),
                    lambda_stab=float(lam),
                    stab_normalize=bool(stab_norm),
                    stab_margin=float(STAB_MARGIN),
                    eval_every=int(EVAL_EVERY),
                    stab_every=int(STAB_EVERY),
                    print_every_sec=float(PRINT_EVERY_SEC),
                    min_print_iter_gap=int(MIN_PRINT_ITER_GAP),
                    print_on_improve_only=bool(PRINT_ON_IMPROVE_ONLY),
                    print_run_start_done=bool(PRINT_RUN_START_DONE),
                )
                res["tag"] = tag
                all_rows.append(res)

                print(f"[progress] {rid}/{total_runs} done. final={res['ratio_final_%']:.6f}% rocket_s={res['t_rocket_s']:.2f}")

    # Save summaries
    df = pd.DataFrame(all_rows)
    all_csv = os.path.join(out_dir, f"{base}_YIANNIS_ROCKET_ONLY_ALL_RUNS.csv")
    df.to_csv(all_csv, index=False)

    grp_cols = ["objective", "stab_normalize", "lambda_stab", "stab_margin", "eval_every", "stab_every"]
    agg = df.groupby(grp_cols).agg(
        mean_final_ratio=("ratio_final_%", "mean"),
        std_final_ratio=("ratio_final_%", "std"),
        mean_rocket_s=("t_rocket_s", "mean"),
        mean_total_s=("t_total_s", "mean"),
        mean_stab_time_s=("stab_time_s", "mean"),
        mean_eval_time_s=("eval_time_s", "mean"),
        mean_base_time_s=("base_time_s", "mean"),
        runs=("ratio_final_%", "count"),
    ).reset_index()

    agg_csv = os.path.join(out_dir, f"{base}_YIANNIS_ROCKET_ONLY_AGG.csv")
    agg.to_csv(agg_csv, index=False)

    sweep_s = time.time() - t_sweep0
    print("------------------------------------------------------------------")
    print(f"[DONE] sweep_total_s={sweep_s:.2f}")
    print(f"ALL runs CSV: {all_csv}")
    print(f"AGG summary:  {agg_csv}")
    print("------------------------------------------------------------------")

    topk = agg.sort_values("mean_final_ratio", ascending=False).head(20)
    with pd.option_context("display.max_rows", 50, "display.max_columns", 50, "display.width", 180):
        print(topk.to_string(index=False))

    print("\nNEXT:")
    print("  1) Pick best (objective, lambda) from AGG.")
    print("  2) Re-run ONLY that config with SEEDS=[1,2,3,4,5] to confirm.\n")



================= YIANNIS PROTOCOL (ROCKET-ONLY) =================
Input:      /mmfs1/home/sv96/Feedback-arc-set-paper/datasets/connectome.d
Out dir:    /mmfs1/home/sv96/Feedback-arc-set-paper/datasets/yiannis_rocket_only_outputs
Loaded:     n=136648 m=5657719 total_w=41912140.000000 load_s=14.98
Budget:     time_limit_s=120.0  beta=1.0  lr=0.05
Speed:      eval_every=1000  stab_every=5
Logs:       print_every_sec=10.0  min_print_iter_gap=300  improve_only=1
Seeds:      [1, 2, 3]
Lambdas:    [0.0, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0] (baseline forces lambda=0)
------------------------------------------------------------------

===== RUN 1/51 : obj=baseline, lam=0.0, seed=1 =====

[run] START obj=baseline lam=0 seed=1 budget=120.0s eval_every=1000 stab_every=5
[rocket] ... obj=baseline lam=0 it=300 ips=9.9 best=49.8280% rem=89.7s stab_calls=0 stab_avg=0.00ms eval_avg=0.00ms L_stab(last)=0
[rocket] ... obj=baseline lam=0 it=600 ips=10.0 best=49.8280% rem=59.7s stab_calls=0 stab_avg=0.0

KeyboardInterrupt: 

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
YIANNIS PROTOCOL — ROCKET-ONLY stability objectives (Ioannis options) + lambda sweep

Protocol (per Yiannis email):
  - Run on connectome
  - 2 min limit per experiment
  - For each added objective, sweep lambda over many values (not just 0.5)
  - Also optionally consider trainable lambda (dual-style update)
  - Save all solutions

Objectives implemented:
  - baseline: Rocket paper objective (maximize soft forward reward)
  - hinge2:  Option A (hinge-squared stability violations)
  - softplus: Option B (softplus stability barrier)
  - swap:    Option C (linear ReLU "swap-regret" penalty)

Speed notes:
  - baseline per iter: O(m) for sigmoid + one scatter (np.add.at)
  - stability: adds multiple bincount reductions over m edges on "stab steps"
    => much slower when lambda>0
  - We throttle:
      * stab_every: compute stability penalty/grad every K iters
      * eval_every: compute discrete FW checkpoint every K iters (argsort cost)
      * printing: time-based + min-iter-gap + improve-only option

Dependencies: numpy, pandas
"""

import os
import time
from collections import defaultdict
import numpy as np
import pandas as pd


# ============================================================
# 0) Helpers
# ============================================================

def safe_mkdir(path: str):
    os.makedirs(path, exist_ok=True)
    return path

def fmt_lam(lam: float):
    s = f"{lam:.6g}"
    return s.replace(".", "p").replace("-", "m")

def ratio_percent(fw, total_w):
    return (100.0 * fw / total_w) if total_w > 0 else 0.0

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def softplus(x):
    # stable softplus
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0.0)

def write_ranking_csv_nodeid_order(path, index_to_node, rank_arr):
    n = len(rank_arr)
    rows = [{"Node ID": str(index_to_node[i]).strip(), "Order": int(rank_arr[i])} for i in range(n)]
    rows.sort(key=lambda r: r["Order"])
    pd.DataFrame(rows).to_csv(path, index=False)

def compute_forward_backward_from_arrays(eu, ev, ew, rank_arr):
    total_w = float(np.sum(ew))
    fw = float(np.sum(ew[rank_arr[eu] < rank_arr[ev]]))
    bw = total_w - fw
    return total_w, fw, bw


# ============================================================
# 1) DIMACS reader (aggregates parallel arcs) -> deterministic
# ============================================================

def read_graph_dimacs_agg(file_path: str):
    agg = defaultdict(float)
    node_ids = set()

    with open(file_path, "r") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith(("c", "p")):
                continue
            if not line.startswith("a"):
                continue

            parts = line.split()
            if len(parts) < 4:
                continue

            u = parts[1]
            v = parts[2]
            try:
                w = float(parts[3])
            except ValueError:
                continue

            node_ids.add(u)
            node_ids.add(v)
            agg[(u, v)] += w

    node_list = sorted(node_ids)
    node_to_index = {node: i for i, node in enumerate(node_list)}
    index_to_node = {i: node for node, i in node_to_index.items()}

    edges_indexed = [(node_to_index[u], node_to_index[v], float(w_sum))
                     for (u, v), w_sum in agg.items()]
    edges_indexed.sort(key=lambda e: (e[0], e[1]))
    return edges_indexed, node_to_index, index_to_node


# ============================================================
# 2) Stability penalty (Ioannis options) + edge dL/dd
#    d_e := P[v] - P[u]
#
# Returns:
#   L_stab: scalar penalty
#   edge_dLd_d: float32 array length m, derivative wrt d_e
#   viol_sum: scalar measure of violation magnitude (for "trainable lambda" dual update)
# ============================================================

def stability_loss_and_edge_dLd_d_bincount(
    n: int,
    eu: np.ndarray, ev: np.ndarray, ew: np.ndarray,
    P: np.ndarray,
    beta: float,
    loss_type: str,          # "hinge2" | "softplus" | "swap"
    normalize: bool,
    Tot: np.ndarray | None,  # precomputed Tot(x)=sum incident weights, float64, length n (only if normalize)
    eps_norm: float = 1e-12,
    margin: float = 0.0,     # only for softplus; acts like slack m
):
    # d_e = P[v] - P[u]
    d = (P[ev] - P[eu]).astype(np.float32)
    x = (float(beta) * d).astype(np.float32)

    fwd = sigmoid(x).astype(np.float32)
    bwd = (1.0 - fwd).astype(np.float32)

    w64 = ew.astype(np.float64)

    # Soft aggregates
    BackIn  = np.bincount(ev, weights=w64 * bwd.astype(np.float64), minlength=n)
    FwdOut  = np.bincount(eu, weights=w64 * fwd.astype(np.float64), minlength=n)
    BackOut = np.bincount(eu, weights=w64 * bwd.astype(np.float64), minlength=n)
    FwdIn   = np.bincount(ev, weights=w64 * fwd.astype(np.float64), minlength=n)

    # Violations
    v1 = BackIn - FwdOut   # want <=0
    v2 = BackOut - FwdIn   # want <=0

    if normalize:
        if Tot is None:
            # (Should not happen if caller precomputes Tot)
            Tot = np.bincount(eu, weights=w64, minlength=n) + np.bincount(ev, weights=w64, minlength=n)
        denom = Tot + float(eps_norm)
        v1n = v1 / denom
        v2n = v2 / denom
    else:
        denom = None
        v1n, v2n = v1, v2

    # Choose penalty & vertex derivatives
    if loss_type == "hinge2":
        r1 = np.maximum(v1n, 0.0)
        r2 = np.maximum(v2n, 0.0)
        viol_sum = float(np.sum(r1) + np.sum(r2))
        L = float(np.sum(r1 * r1) + np.sum(r2 * r2))

        dv1n = (2.0 * r1).astype(np.float64)
        dv2n = (2.0 * r2).astype(np.float64)

        if normalize:
            dv1 = dv1n / denom
            dv2 = dv2n / denom
        else:
            dv1, dv2 = dv1n, dv2n

    elif loss_type == "swap":
        # Option C: linear ReLU
        r1 = np.maximum(v1n, 0.0)
        r2 = np.maximum(v2n, 0.0)
        viol_sum = float(np.sum(r1) + np.sum(r2))
        L = float(np.sum(r1) + np.sum(r2))

        dv1n = (v1n > 0.0).astype(np.float64)
        dv2n = (v2n > 0.0).astype(np.float64)

        if normalize:
            dv1 = dv1n / denom
            dv2 = dv2n / denom
        else:
            dv1, dv2 = dv1n, dv2n

    elif loss_type == "softplus":
        z1 = v1n - float(margin)
        z2 = v2n - float(margin)
        # use sigmoid(z) as derivative of softplus
        s1 = sigmoid(z1).astype(np.float64)
        s2 = sigmoid(z2).astype(np.float64)
        # violation magnitude for dual update: use softplus arguments' positive part as proxy
        viol_sum = float(np.sum(np.maximum(z1, 0.0)) + np.sum(np.maximum(z2, 0.0)))
        L = float(np.sum(softplus(z1)) + np.sum(softplus(z2)))

        if normalize:
            dv1 = s1 / denom
            dv2 = s2 / denom
        else:
            dv1, dv2 = s1, s2

    else:
        raise ValueError(f"Unknown stability loss_type={loss_type}")

    # Vertex coeffs for aggregates
    # v1 = BackIn - FwdOut
    g_BackIn  = dv1
    g_FwdOut  = -dv1
    # v2 = BackOut - FwdIn
    g_BackOut = dv2
    g_FwdIn   = -dv2

    # df/dd = beta*fwd*(1-fwd); db/dd = -df/dd
    df_dd = (float(beta) * (fwd * (1.0 - fwd))).astype(np.float32)
    db_dd = (-df_dd).astype(np.float32)

    # edge derivative wrt d_e
    # BackIn depends on edges into v; BackOut depends on edges out of u
    # FwdOut depends on edges out of u; FwdIn depends on edges into v
    edge_dLd_d = (
        (ew * db_dd) * (g_BackIn[ev].astype(np.float32) + g_BackOut[eu].astype(np.float32)) +
        (ew * df_dd) * (g_FwdOut[eu].astype(np.float32) + g_FwdIn[ev].astype(np.float32))
    ).astype(np.float32)

    return L, edge_dLd_d, viol_sum


# ============================================================
# 3) Rocket optimizer (baseline + optional stability term)
#
# IMPORTANT: baseline path is kept minimal: no Tot precompute, no stability work.
# ============================================================

def rocket_optimize(
    n: int,
    eu: np.ndarray, ev: np.ndarray, ew: np.ndarray,
    *,
    beta: float,
    lr: float,
    seed: int,
    deadline: float,
    objective: str,              # "baseline" | "hinge2" | "softplus" | "swap"
    lambda_mode: str,            # "fixed" | "dual"   (dual only meaningful for non-baseline)
    lambda_init: float,
    lambda_eta: float,           # dual step size
    lambda_max: float,           # dual cap
    stab_normalize: bool,
    stab_margin: float,
    total_w: float,
    eval_every: int,
    stab_every: int,
    # Logging throttle:
    print_every_sec: float,
    min_print_iter_gap: int,
    improve_only: bool,
):
    rng = np.random.default_rng(seed)
    P = rng.standard_normal(n).astype(np.float32)

    # weight max-normalization (Rocket paper style)
    wmax = float(np.max(ew)) if ew.size else 1.0
    if wmax <= 0:
        wmax = 1.0
    w_hat = (ew / wmax).astype(np.float32)

    # Adam state
    m_adam = np.zeros(n, dtype=np.float32)
    v_adam = np.zeros(n, dtype=np.float32)
    b1, b2 = 0.9, 0.999
    eps = 1e-8
    t = 0

    def rank_from_positions(Pvec):
        perm = np.argsort(Pvec, kind="mergesort").astype(np.int32)
        r = np.empty(n, dtype=np.int32)
        r[perm] = np.arange(n, dtype=np.int32)
        return r

    # init best by discrete FW
    rank0 = rank_from_positions(P)
    fw_best = float(np.sum(ew[rank0[eu] < rank0[ev]]))
    rank_best = rank0.copy()

    # Tot precompute ONLY if needed
    Tot = None
    if objective != "baseline" and stab_normalize:
        w64 = ew.astype(np.float64)
        Tot = np.bincount(eu, weights=w64, minlength=n) + np.bincount(ev, weights=w64, minlength=n)

    lam = float(lambda_init)

    # Stats
    it = 0
    last_print_t = time.time()
    last_print_it = 0
    last_print_best = fw_best

    stab_calls = 0
    stab_time = 0.0
    eval_calls = 1
    eval_time = 0.0

    # last computed values for printing
    L_stab_last = 0.0
    viol_last = 0.0

    while time.time() < deadline:
        it += 1

        # baseline per-iter terms
        d = (P[ev] - P[eu]).astype(np.float32)
        x = (float(beta) * d).astype(np.float32)
        sig_f = sigmoid(x).astype(np.float32)
        sigp = (sig_f * (1.0 - sig_f)).astype(np.float32)

        # baseline gradient contribution (maximize reward => minimize -reward)
        edge_grad_u = (w_hat * sigp * float(beta)).astype(np.float32)

        # Stability term (throttled)
        if objective != "baseline" and lam != 0.0:
            if stab_every <= 1 or (it % int(stab_every) == 0):
                ts0 = time.time()
                L_stab, edge_dLd_d, viol_sum = stability_loss_and_edge_dLd_d_bincount(
                    n=n, eu=eu, ev=ev, ew=ew,
                    P=P, beta=float(beta),
                    loss_type=str(objective),
                    normalize=bool(stab_normalize),
                    Tot=Tot,
                    margin=float(stab_margin),
                )
                stab_time += (time.time() - ts0)
                stab_calls += 1
                L_stab_last = float(L_stab)
                viol_last = float(viol_sum)

                # Add stability gradient: maximize (reward - lam * L_stab)
                # loss = -reward + lam*L_stab  => grad(loss) = -grad(reward) + lam*grad(L_stab)
                # our edge_grad_u is grad(loss) w.r.t u-sign convention already (+ at u, - at v)
                edge_grad_u = (edge_grad_u - lam * edge_dLd_d).astype(np.float32)

                # Optional trainable lambda (dual-ish): increase lam if violations persist
                if lambda_mode == "dual":
                    # push lambda up when viol_sum is positive; cap it
                    lam = min(float(lambda_max), max(0.0, lam + float(lambda_eta) * float(viol_sum)))

            # else: skip stability this iter (use last lam)
        else:
            L_stab_last = 0.0
            viol_last = 0.0

        # Scatter to node gradient + Adam update
        grad = np.zeros(n, dtype=np.float32)
        np.add.at(grad, eu, +edge_grad_u)
        np.add.at(grad, ev, -edge_grad_u)

        t += 1
        m_adam = (b1 * m_adam + (1 - b1) * grad).astype(np.float32)
        v_adam = (b2 * v_adam + (1 - b2) * (grad * grad)).astype(np.float32)
        mhat = m_adam / (1 - (b1 ** t))
        vhat = v_adam / (1 - (b2 ** t))
        P = (P - float(lr) * mhat / (np.sqrt(vhat) + eps)).astype(np.float32)

        # Discrete checkpoint (argsort + FW)
        if eval_every and (it % int(eval_every) == 0):
            te0 = time.time()
            rank = rank_from_positions(P)
            fw = float(np.sum(ew[rank[eu] < rank[ev]]))
            eval_time += (time.time() - te0)
            eval_calls += 1
            if fw > fw_best + 1e-9:
                fw_best = fw
                rank_best = rank.copy()

        # Print throttling
        now = time.time()
        improved_since_print = (fw_best > last_print_best + 1e-9)
        enough_time = (now - last_print_t) >= float(print_every_sec)
        enough_iters = (it - last_print_it) >= int(min_print_iter_gap)

        should_print = False
        if improve_only:
            should_print = improved_since_print and enough_iters
            # also allow occasional heartbeat if a lot of time passed
            if not should_print and enough_time and enough_iters:
                should_print = True
        else:
            should_print = enough_time and enough_iters

        if should_print:
            rem = max(0.0, float(deadline - now))
            best_ratio = ratio_percent(fw_best, total_w)
            eval_avg_ms = (eval_time / max(1, eval_calls)) * 1000.0
            stab_avg_ms = (stab_time / max(1, stab_calls)) * 1000.0 if stab_calls else 0.0
            tag = "IMPROVED" if improved_since_print else "..."
            lam_show = lam if (objective != "baseline") else 0.0
            print(
                f"[rocket] {tag} obj={objective:<8} lam={lam_show:g} it={it} "
                f"best={best_ratio:.4f}% rem={rem:.1f}s "
                f"stab_calls={stab_calls} stab_avg={stab_avg_ms:.2f}ms "
                f"eval_avg={eval_avg_ms:.2f}ms "
                f"L_stab(last)={L_stab_last:.3g} viol(last)={viol_last:.3g}"
            )
            last_print_t = now
            last_print_it = it
            last_print_best = fw_best

    # Final checkpoint
    rank = rank_from_positions(P)
    fw = float(np.sum(ew[rank[eu] < rank[ev]]))
    if fw > fw_best + 1e-9:
        fw_best = fw
        rank_best = rank.copy()

    stats = {
        "iters_done": int(it),
        "stab_calls": int(stab_calls),
        "stab_time_s": float(stab_time),
        "eval_calls": int(eval_calls),
        "eval_time_s": float(eval_time),
        "lambda_final": float(lam),
    }
    return rank_best, fw_best, stats


# ============================================================
# 4) One run wrapper
# ============================================================

def run_one_config(
    *,
    dimacs_path: str,
    index_to_node,
    n: int,
    eu, ev, ew,
    out_csv_path: str,
    seed: int,
    time_limit_s: float,
    rocket_beta: float,
    rocket_lr: float,
    objective: str,
    lambda_mode: str,
    lambda_init: float,
    lambda_eta: float,
    lambda_max: float,
    stab_normalize: bool,
    stab_margin: float,
    eval_every: int,
    stab_every: int,
    print_every_sec: float,
    min_print_iter_gap: int,
    improve_only: bool,
):
    t0 = time.time()
    deadline = t0 + float(time_limit_s)
    total_w = float(np.sum(ew))

    print(
        f"\n[run] START obj={objective} lam_mode={lambda_mode} lam0={lambda_init:g} seed={seed} "
        f"budget={time_limit_s}s eval_every={eval_every} stab_every={stab_every}"
    )

    rank_best, fw_best, stats = rocket_optimize(
        n=int(n), eu=eu, ev=ev, ew=ew,
        beta=float(rocket_beta),
        lr=float(rocket_lr),
        seed=int(seed),
        deadline=float(deadline),
        objective=str(objective),
        lambda_mode=str(lambda_mode),
        lambda_init=float(lambda_init),
        lambda_eta=float(lambda_eta),
        lambda_max=float(lambda_max),
        stab_normalize=bool(stab_normalize),
        stab_margin=float(stab_margin),
        total_w=float(total_w),
        eval_every=int(eval_every),
        stab_every=int(stab_every),
        print_every_sec=float(print_every_sec),
        min_print_iter_gap=int(min_print_iter_gap),
        improve_only=bool(improve_only),
    )

    tot1, fw1, bw1 = compute_forward_backward_from_arrays(eu, ev, ew, rank_best)
    write_ranking_csv_nodeid_order(out_csv_path, index_to_node, rank_best)

    elapsed = time.time() - t0
    final_ratio = ratio_percent(float(fw1), float(tot1))

    print(
        f"[run] DONE  obj={objective} lam_mode={lambda_mode} lam_final={stats['lambda_final']:.6g} seed={seed} "
        f"final={final_ratio:.6f}% total_s={elapsed:.2f} iters={stats['iters_done']} "
        f"stab_calls={stats['stab_calls']} stab_time_s={stats['stab_time_s']:.2f} "
        f"eval_time_s={stats['eval_time_s']:.2f} csv={out_csv_path}"
    )

    return {
        "seed": int(seed),
        "n": int(n),
        "m": int(eu.size),
        "objective": str(objective),
        "lambda_mode": str(lambda_mode),
        "lambda_init": float(lambda_init),
        "lambda_eta": float(lambda_eta),
        "lambda_max": float(lambda_max),
        "lambda_final": float(stats["lambda_final"]),
        "stab_normalize": bool(stab_normalize),
        "stab_margin": float(stab_margin),
        "eval_every": int(eval_every),
        "stab_every": int(stab_every),

        "ratio_final_%": float(final_ratio),
        "final_fw": float(fw1),
        "final_bw": float(bw1),
        "total_w": float(tot1),

        "iters_done": int(stats["iters_done"]),
        "t_total_s": float(elapsed),

        "stab_calls": int(stats["stab_calls"]),
        "stab_time_s": float(stats["stab_time_s"]),
        "eval_calls": int(stats["eval_calls"]),
        "eval_time_s": float(stats["eval_time_s"]),

        "out_csv": out_csv_path,
    }


# ============================================================
# 5) Main — Yiannis protocol sweep
# ============================================================

if __name__ == "__main__":
    # ------------------------------
    # INPUT
    # ------------------------------
    edge_file = "/mmfs1/home/sv96/Feedback-arc-set-paper/datasets/connectome.d"

    # ------------------------------
    # OUTPUT
    # ------------------------------
    desktop = os.path.join(os.path.expanduser("~"), "Desktop")
    out_root = desktop if os.path.isdir(desktop) else os.path.dirname(os.path.abspath(edge_file))
    out_dir = safe_mkdir(os.path.join(out_root, "yiannis_rocket_only_outputs"))

    base = os.path.basename(edge_file)
    base = base[:-2] if base.endswith(".d") else base

    # ------------------------------
    # Protocol knobs (Yiannis)
    # ------------------------------
    TIME_LIMIT_S = 120.0  # 2 min/run
    SEEDS = [1, 2, 3]     # screening; later use [1,2,3,4,5]

    # Rocket hyperparams (keep consistent)
    ROCKET_BETA = 1.0
    ROCKET_LR = 0.05

    # Speed knobs
    # - eval_every triggers argsort+FW; too frequent adds overhead
    # - stab_every triggers expensive stability bincount work; 5 is a good compromise
    EVAL_EVERY = 1000
    STAB_EVERY = 5

    # Screen logging knobs (less spam)
    PRINT_EVERY_SEC = 15.0
    MIN_PRINT_ITER_GAP = 800
    IMPROVE_ONLY = True

    # Stability settings
    STAB_NORMALIZE = True
    STAB_MARGIN = 0.0  # only affects softplus

    # Lambda sweep (fixed)
    LAM_LIST = [0.0, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]

    # Trainable lambda (dual-style) — optional extra variant(s)
    # This is NOT the same as "learn lambda by minimizing the same objective" (which would push lambda -> 0).
    # Instead it behaves like a Lagrange-multiplier update: if violations persist, lambda increases.
    DO_DUAL_LAMBDA_RUNS = True
    DUAL_LAM0 = 0.01
    DUAL_ETA = 1e-6
    DUAL_LAM_MAX = 100.0

    # Objectives to test (Ioannis/Yiannis options)
    # baseline is always run with lambda=0
    OBJECTIVES = ["baseline", "hinge2", "softplus", "swap"]

    # ------------------------------
    # Load graph once
    # ------------------------------
    t_load0 = time.time()
    edges_indexed, node_to_index, index_to_node = read_graph_dimacs_agg(edge_file)
    n = len(node_to_index)
    m = len(edges_indexed)

    eu = np.empty(m, dtype=np.int32)
    ev = np.empty(m, dtype=np.int32)
    ew = np.empty(m, dtype=np.float32)
    for i, (u, v, w) in enumerate(edges_indexed):
        eu[i] = u
        ev[i] = v
        ew[i] = float(w)

    total_w = float(np.sum(ew))
    load_s = time.time() - t_load0

    print("\n================= YIANNIS PROTOCOL (ROCKET-ONLY) =================")
    print(f"Input:      {edge_file}")
    print(f"Out dir:    {out_dir}")
    print(f"Loaded:     n={n} m={m} total_w={total_w:.6f} load_s={load_s:.2f}")
    print(f"Budget:     time_limit_s={TIME_LIMIT_S}  beta={ROCKET_BETA}  lr={ROCKET_LR}")
    print(f"Speed:      eval_every={EVAL_EVERY}  stab_every={STAB_EVERY}")
    print(f"Logs:       print_every_sec={PRINT_EVERY_SEC}  min_print_iter_gap={MIN_PRINT_ITER_GAP}  improve_only={int(IMPROVE_ONLY)}")
    print(f"Seeds:      {SEEDS}")
    print(f"Lambdas:    {LAM_LIST} (baseline forces lambda=0)")
    if DO_DUAL_LAMBDA_RUNS:
        print(f"Dual-lambda: enabled (lam0={DUAL_LAM0}, eta={DUAL_ETA}, lam_max={DUAL_LAM_MAX})")
    print("------------------------------------------------------------------")

    # Count runs
    total_runs = 0
    for obj in OBJECTIVES:
        if obj == "baseline":
            total_runs += len(SEEDS)
        else:
            total_runs += len(SEEDS) * len(LAM_LIST)
            if DO_DUAL_LAMBDA_RUNS:
                total_runs += len(SEEDS)  # one dual run per objective
    print(f"Planned runs: {total_runs}")
    print("------------------------------------------------------------------")

    all_rows = []
    rid = 0
    t_sweep0 = time.time()

    for objective in OBJECTIVES:
        if objective == "baseline":
            lam_variants = [("fixed", 0.0)]
        else:
            lam_variants = [("fixed", lam) for lam in LAM_LIST]
            if DO_DUAL_LAMBDA_RUNS:
                lam_variants.append(("dual", DUAL_LAM0))

        for lambda_mode, lam0 in lam_variants:
            for seed in SEEDS:
                rid += 1

                # Tag + output filename
                if objective == "baseline":
                    tag = f"{objective}_lam0_seed{seed}"
                else:
                    if lambda_mode == "fixed":
                        tag = f"{objective}_fixed_norm{int(STAB_NORMALIZE)}_lam{fmt_lam(lam0)}_seed{seed}"
                    else:
                        tag = f"{objective}_dual_norm{int(STAB_NORMALIZE)}_lam0_{fmt_lam(lam0)}_eta{fmt_lam(DUAL_ETA)}_seed{seed}"

                out_csv = os.path.join(out_dir, f"{base}_Seke_{tag}_ranking.csv")

                print(f"\n===== RUN {rid}/{total_runs} : obj={objective}, mode={lambda_mode}, lam0={lam0}, seed={seed} =====")

                res = run_one_config(
                    dimacs_path=edge_file,
                    index_to_node=index_to_node,
                    n=n,
                    eu=eu, ev=ev, ew=ew,
                    out_csv_path=out_csv,
                    seed=int(seed),
                    time_limit_s=float(TIME_LIMIT_S),
                    rocket_beta=float(ROCKET_BETA),
                    rocket_lr=float(ROCKET_LR),

                    objective=str(objective),
                    lambda_mode=str(lambda_mode),
                    lambda_init=float(lam0),
                    lambda_eta=float(DUAL_ETA if lambda_mode == "dual" else 0.0),
                    lambda_max=float(DUAL_LAM_MAX if lambda_mode == "dual" else 0.0),

                    stab_normalize=bool(STAB_NORMALIZE),
                    stab_margin=float(STAB_MARGIN),

                    eval_every=int(EVAL_EVERY),
                    stab_every=int(STAB_EVERY),

                    print_every_sec=float(PRINT_EVERY_SEC),
                    min_print_iter_gap=int(MIN_PRINT_ITER_GAP),
                    improve_only=bool(IMPROVE_ONLY),
                )

                res["tag"] = tag
                all_rows.append(res)
                print(f"[progress] {rid}/{total_runs} done. final={res['ratio_final_%']:.6f}%")

    df = pd.DataFrame(all_rows)

    all_csv = os.path.join(out_dir, f"{base}_YIANNIS_ROCKET_ONLY_ALL_RUNS.csv")
    df.to_csv(all_csv, index=False)

    grp_cols = [
        "objective", "lambda_mode", "stab_normalize", "lambda_init", "lambda_eta", "lambda_max",
        "stab_margin", "eval_every", "stab_every"
    ]
    agg = df.groupby(grp_cols).agg(
        mean_final_ratio=("ratio_final_%", "mean"),
        std_final_ratio=("ratio_final_%", "std"),
        mean_total_s=("t_total_s", "mean"),
        mean_stab_time_s=("stab_time_s", "mean"),
        mean_eval_time_s=("eval_time_s", "mean"),
        runs=("ratio_final_%", "count"),
    ).reset_index()

    agg_csv = os.path.join(out_dir, f"{base}_YIANNIS_ROCKET_ONLY_AGG.csv")
    agg.to_csv(agg_csv, index=False)

    sweep_s = time.time() - t_sweep0
    print("------------------------------------------------------------------")
    print(f"[DONE] sweep_total_s={sweep_s:.2f}")
    print(f"ALL runs CSV: {all_csv}")
    print(f"AGG summary:  {agg_csv}")
    print("------------------------------------------------------------------")

    topk = agg.sort_values("mean_final_ratio", ascending=False).head(20)
    with pd.option_context("display.max_rows", 50, "display.max_columns", 80, "display.width", 200):
        print(topk.to_string(index=False))

    print("\nNEXT (Yiannis-style):")
    print("  1) Pick the best row from AGG (objective + mode + lambda).")
    print("  2) Re-run ONLY that config with SEEDS=[1,2,3,4,5] to confirm.\n")



================= YIANNIS PROTOCOL (ROCKET-ONLY) =================
Input:      /mmfs1/home/sv96/Feedback-arc-set-paper/datasets/connectome.d
Out dir:    /mmfs1/home/sv96/Feedback-arc-set-paper/datasets/yiannis_rocket_only_outputs
Loaded:     n=136648 m=5657719 total_w=41912140.000000 load_s=13.14
Budget:     time_limit_s=120.0  beta=1.0  lr=0.05
Speed:      eval_every=1000  stab_every=5
Logs:       print_every_sec=15.0  min_print_iter_gap=800  improve_only=1
Seeds:      [1, 2, 3]
Lambdas:    [0.0, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0] (baseline forces lambda=0)
Dual-lambda: enabled (lam0=0.01, eta=1e-06, lam_max=100.0)
------------------------------------------------------------------
Planned runs: 84
------------------------------------------------------------------

===== RUN 1/84 : obj=baseline, mode=fixed, lam0=0.0, seed=1 =====

[run] START obj=baseline lam_mode=fixed lam0=0 seed=1 budget=120.0s eval_every=1000 stab_every=5
[rocket] ... obj=baseline lam=0 it=800 best=49.8280% rem

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Run Rocket twice on the same DIMACS graph from two different random starts,
then measure how similar the two solutions are in terms of edges.

We treat a "solution" as a ranking (order) over vertices.
An edge (u->v) is "forward" under a ranking if rank[u] < rank[v].

We report:
  1) Forward-edge overlap (Jaccard): |F1 ∩ F2| / |F1 ∪ F2|
  2) Both-forward fraction over ALL edges: |F1 ∩ F2| / m
  3) Full edge-direction agreement: fraction of edges where (forward/backward) matches

Dependencies: numpy
"""

import time
from collections import defaultdict
import numpy as np


# ============================================================
# Helpers
# ============================================================

def sigmoid(x):
    # numerically safe sigmoid
    x = np.clip(x, -50.0, 50.0)
    return 1.0 / (1.0 + np.exp(-x))


# ============================================================
# DIMACS reader (aggregates parallel arcs) -> deterministic
# ============================================================

def read_graph_dimacs_agg(file_path: str):
    agg = defaultdict(float)
    node_ids = set()

    with open(file_path, "r") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith(("c", "p")):
                continue
            if not line.startswith("a"):
                continue

            parts = line.split()
            if len(parts) < 4:
                continue

            u = parts[1]
            v = parts[2]
            try:
                w = float(parts[3])
            except ValueError:
                continue

            node_ids.add(u)
            node_ids.add(v)
            agg[(u, v)] += w

    node_list = sorted(node_ids)
    node_to_index = {node: i for i, node in enumerate(node_list)}
    edges_indexed = [(node_to_index[u], node_to_index[v], float(w_sum))
                     for (u, v), w_sum in agg.items()]
    edges_indexed.sort(key=lambda e: (e[0], e[1]))
    return edges_indexed, node_to_index


# ============================================================
# Rocket (baseline only): Adam on smooth forward objective
# ============================================================

def rocket_baseline(
    n: int,
    eu: np.ndarray, ev: np.ndarray, ew: np.ndarray,
    *,
    beta: float,
    lr: float,
    seed: int,
    time_limit_s: float,
    eval_every: int = 1000,
):
    rng = np.random.default_rng(seed)
    P = rng.standard_normal(n).astype(np.float32)

    # max-normalize weights (paper Eq. 7)
    wmax = float(np.max(ew)) if ew.size else 1.0
    if wmax <= 0:
        wmax = 1.0
    w_hat = (ew / wmax).astype(np.float32)

    # Adam state
    m_adam = np.zeros(n, dtype=np.float32)
    v_adam = np.zeros(n, dtype=np.float32)
    b1, b2 = 0.9, 0.999
    eps = 1e-8
    t = 0

    def rank_from_positions(Pvec):
        perm = np.argsort(Pvec, kind="mergesort").astype(np.int32)
        r = np.empty(n, dtype=np.int32)
        r[perm] = np.arange(n, dtype=np.int32)
        return r

    # best discrete seen so far
    rank_best = rank_from_positions(P)
    fw_best = float(np.sum(ew[rank_best[eu] < rank_best[ev]]))

    deadline = time.time() + float(time_limit_s)
    it = 0

    while time.time() < deadline:
        it += 1

        d = (P[ev] - P[eu]).astype(np.float32)
        x = (float(beta) * d).astype(np.float32)
        sig_f = sigmoid(x).astype(np.float32)
        sigp = (sig_f * (1.0 - sig_f)).astype(np.float32)

        # edge contribution (baseline)
        edge_grad_u = (w_hat * sigp * float(beta)).astype(np.float32)

        # faster scatter via bincount
        grad = (
            np.bincount(eu, weights=edge_grad_u.astype(np.float64), minlength=n)
            - np.bincount(ev, weights=edge_grad_u.astype(np.float64), minlength=n)
        ).astype(np.float32)

        # Adam step
        t += 1
        m_adam = (b1 * m_adam + (1 - b1) * grad).astype(np.float32)
        v_adam = (b2 * v_adam + (1 - b2) * (grad * grad)).astype(np.float32)
        mhat = m_adam / (1 - (b1 ** t))
        vhat = v_adam / (1 - (b2 ** t))
        P = (P - float(lr) * mhat / (np.sqrt(vhat) + eps)).astype(np.float32)

        # discrete checkpoint
        if eval_every and (it % int(eval_every) == 0):
            rank = rank_from_positions(P)
            fw = float(np.sum(ew[rank[eu] < rank[ev]]))
            if fw > fw_best + 1e-9:
                fw_best = fw
                rank_best = rank.copy()

    # final checkpoint
    rank = rank_from_positions(P)
    fw = float(np.sum(ew[rank[eu] < rank[ev]]))
    if fw > fw_best + 1e-9:
        fw_best = fw
        rank_best = rank.copy()

    return rank_best, fw_best, it


# ============================================================
# Similarity metrics
# ============================================================

def edge_forward_mask(eu, ev, rank):
    return rank[eu] < rank[ev]  # True if edge is forward under ranking


def compare_two_rankings(eu, ev, rank1, rank2):
    f1 = edge_forward_mask(eu, ev, rank1)
    f2 = edge_forward_mask(eu, ev, rank2)

    both_forward = int(np.count_nonzero(f1 & f2))
    either_forward = int(np.count_nonzero(f1 | f2))
    agree_direction = int(np.count_nonzero(f1 == f2))
    m = int(eu.size)

    jaccard = (both_forward / either_forward) if either_forward > 0 else 1.0
    both_over_all = both_forward / m if m > 0 else 0.0
    agree_over_all = agree_direction / m if m > 0 else 0.0

    return {
        "m": m,
        "both_forward": both_forward,
        "either_forward": either_forward,
        "agree_direction": agree_direction,
        "forward_overlap_jaccard_%": 100.0 * jaccard,
        "both_forward_over_all_edges_%": 100.0 * both_over_all,
        "direction_agreement_over_all_edges_%": 100.0 * agree_over_all,
    }


# ============================================================
# Main (hardcoded input path)
# ============================================================

if __name__ == "__main__":
    edge_file = "/mmfs1/home/sv96/Feedback-arc-set-paper/datasets/connectome.d"

    # Hyperparams / budget
    TIME_LIMIT_S = 120.0
    BETA = 1.0
    LR = 0.05
    EVAL_EVERY = 1000

    # Two different random starting points
    SEED1 = 1
    SEED2 = 2

    # -------- Load graph --------
    t0 = time.time()
    edges, node_to_index = read_graph_dimacs_agg(edge_file)
    n = len(node_to_index)
    m = len(edges)

    eu = np.empty(m, dtype=np.int32)
    ev = np.empty(m, dtype=np.int32)
    ew = np.empty(m, dtype=np.float32)
    for i, (u, v, w) in enumerate(edges):
        eu[i] = u
        ev[i] = v
        ew[i] = float(w)

    print(f"Loaded graph: {edge_file}")
    print(f"n={n} m={m} read_s={time.time()-t0:.2f}")

    # -------- Run Rocket twice --------
    print(f"\nRun #1: seed={SEED1}")
    r1, fw1, it1 = rocket_baseline(
        n, eu, ev, ew,
        beta=BETA, lr=LR, seed=SEED1,
        time_limit_s=TIME_LIMIT_S,
        eval_every=EVAL_EVERY,
    )
    print(f"  iters={it1} fw={fw1:.6f}")

    print(f"\nRun #2: seed={SEED2}")
    r2, fw2, it2 = rocket_baseline(
        n, eu, ev, ew,
        beta=BETA, lr=LR, seed=SEED2,
        time_limit_s=TIME_LIMIT_S,
        eval_every=EVAL_EVERY,
    )
    print(f"  iters={it2} fw={fw2:.6f}")

    # -------- Compare solutions --------
    stats = compare_two_rankings(eu, ev, r1, r2)

    print("\n================ Similarity (edge-based) ================")
    print(f"m = {stats['m']}")
    print(f"both_forward = {stats['both_forward']}")
    print(f"either_forward = {stats['either_forward']}")
    print(f"agree_direction = {stats['agree_direction']}")
    print(f"Forward-edge overlap (Jaccard):      {stats['forward_overlap_jaccard_%']:.4f}%")
    print(f"Both-forward over ALL edges:         {stats['both_forward_over_all_edges_%']:.4f}%")
    print(f"Direction agreement (all edges):     {stats['direction_agreement_over_all_edges_%']:.4f}%")
    print("=========================================================")


Loaded graph: /mmfs1/home/sv96/Feedback-arc-set-paper/datasets/connectome.d
n=136648 m=5657719 read_s=11.63

Run #1: seed=1
  iters=1083 fw=34034120.000000

Run #2: seed=2
  iters=1060 fw=34034420.000000

================ Similarity (edge-based) ================
m = 5657719
both_forward = 4239222
either_forward = 4386080
agree_direction = 5510861
Forward-edge overlap (Jaccard):      96.6517%
Both-forward over ALL edges:         74.9281%
Direction agreement (all edges):     97.4043%
